In [1]:
import kagglehub
import pandas as pd
from pathlib import Path
import re


c:\Users\User\Desktop\test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1) Load dataset

In [2]:


# Download latest version
path = kagglehub.dataset_download("mehranrezvani/tehran-house-price-divar")
print("Path to dataset files:", path)
path = Path(path)



Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\mehranrezvani\tehran-house-price-divar\versions\2


In [3]:
# خواندن فایل csv
df = pd.read_csv(path/ "Tehran-Houses-DIVAR.csv")
df


,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD)
0,63,1,True,True,True,Shahran,1850000000,61666.67
1,60,1,True,True,True,Shahran,1850000000,61666.67
2,79,2,True,True,True,Pardis,550000000,18333.33
3,95,2,True,True,True,Shahrake Qods,902500000,30083.33
4,123,2,True,True,True,Shahrake Gharb,7000000000,233333.33
...,...,...,...,...,...,...,...,...
3474,86,2,True,True,True,Southern Janatabad,3500000000,116666.67
3475,83,2,True,True,True,Niavaran,6800000000,226666.67
3476,75,2,False,False,False,Parand,365000000,12166.67
3477,105,2,True,True,True,Dorous,5600000000,186666.67


# 2) Basic sanity checks

In [4]:

print("info:" , df.info())

print("isnull: " , df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3479 entries, 0 to 3478
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Area        3479 non-null   object 
 1   Room        3479 non-null   int64  
 2   Parking     3479 non-null   bool   
 3   Warehouse   3479 non-null   bool   
 4   Elevator    3479 non-null   bool   
 5   Address     3456 non-null   object 
 6   Price       3479 non-null   int64  
 7   Price(USD)  3479 non-null   float64
dtypes: bool(3), float64(1), int64(2), object(2)
memory usage: 146.2+ KB
info: None
isnull:  Area           0
Room           0
Parking        0
Warehouse      0
Elevator       0
Address       23
Price          0
Price(USD)     0
dtype: int64


In [5]:
df = df.drop_duplicates()

# 3) Clean/convert numeric columns

In [6]:
df['Area'] = df['Area'].astype(str).apply(lambda x: re.sub(',', '', x))
df["Area"] = pd.to_numeric(df["Area"], errors='coerce')
df=df.dropna()

C:\Users\User\AppData\Local\Temp\ipykernel_19704\1802639778.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Area'] = df['Area'].astype(str).apply(lambda x: re.sub(',', '', x))
C:\Users\User\AppData\Local\Temp\ipykernel_19704\1802639778.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Area"] = pd.to_numeric(df["Area"], errors='coerce')


In [7]:
print("describe:")
print(df.describe())

describe:
               Area         Room         Price    Price(USD)
count  3.248000e+03  3248.000000  3.248000e+03  3.248000e+03
mean   9.365872e+06     2.088054  5.478118e+09  1.826039e+05
std    3.277906e+08     0.764716  8.267916e+09  2.755972e+05
min    3.000000e+01     0.000000  3.600000e+06  1.200000e+02
25%    7.000000e+01     2.000000  1.420000e+09  4.733333e+04
50%    9.000000e+01     2.000000  2.977500e+09  9.925000e+04
75%    1.220000e+02     2.000000  6.200000e+09  2.066667e+05
max    1.616000e+10     5.000000  9.240000e+10  3.080000e+06


In [8]:
# بزرگ‌ترین Area ها رو ببین
df['Area'].sort_values(ascending=False).head(20)

# رکوردهایی که Area خیلی غیرمنطقی دارن رو جدا کن (مثلاً بالاتر از 5000)
bad = df[df['Area'] > 5000]
bad[['Area', 'Room', 'Price', 'Price(USD)']].head(20)

# تعدادشون؟
bad.shape


(4, 8)

In [9]:
# حذف رکوردهای خراب Area (بالای 3600 یا حتی 5000)
df = df[(df['Area'] >= 30) & (df['Area'] <= 3600)].copy()

# حالا describe باید به اون چیزی که گفتی نزدیک بشه
df[['Area','Room','Price','Price(USD)']].describe()


,Area,Room,Price,Price(USD)
count,3244.000000,3244.000000,3.244000e+03,3.244000e+03
mean,109.052713,2.087855,5.475403e+09,1.825134e+05
std,95.043282,0.765015,8.270445e+09,2.756815e+05
min,30.000000,0.000000,3.600000e+06,1.200000e+02
25%,70.000000,2.000000,1.420000e+09,4.733333e+04
50%,90.000000,2.000000,2.972500e+09,9.908333e+04
75%,122.000000,2.000000,6.181250e+09,2.060417e+05
max,3600.000000,5.000000,9.240000e+10,3.080000e+06


# 4) Convert boolean/yes-no columns to 0/1

In [11]:
def ab (name):
    a=[]
    for i in df[name]:
        if i == True:
            a.append(0)
        elif i == False:
            a.append(1)
    return a

df['Elevator'] = ab('Elevator')
df['Parking'] = ab('Parking')
df['Warehouse'] = ab('Warehouse')

# 5) Clean text column (Address)

In [12]:
if "Address" in df.columns:
    df["Address"] = (
        df["Address"]
        .astype(str)
        .str.strip()                 # حذف فاصله‌های ابتدا/انتها
        .str.replace(r"\s+", " ", regex=True)  # یکسان‌سازی فاصله‌ها
    )

# 6) Handle missing values (NOT global dropna)

In [13]:
# # فقط ستون‌های حیاتی برای مدل قیمت
# df = df.dropna(subset=["Area", "Room", "Price"])

# # برای باینری‌ها اگر null موند، به 0 تبدیل کن (یعنی "ندارد" / نامشخص)
# for col in ["Parking", "Warehouse", "Elevator"]:
#     if col in df.columns:
#         df[col] = df[col].fillna(0).astype(int)


# 7) Remove obvious outliers

In [14]:
df = df[(df["Area"] >= 30) & (df["Area"] <= 3600)]
df = df[(df["Room"] >= 0) & (df["Room"] <= 10)]
df = df[df["Price"] > 0]


# 8) Final output

In [15]:
print("clean shape:", df.shape)
print(df[["Area", "Room", "Price", "Price(USD)"]].describe())

clean shape: (3244, 8)
              Area         Room         Price    Price(USD)
count  3244.000000  3244.000000  3.244000e+03  3.244000e+03
mean    109.052713     2.087855  5.475403e+09  1.825134e+05
std      95.043282     0.765015  8.270445e+09  2.756815e+05
min      30.000000     0.000000  3.600000e+06  1.200000e+02
25%      70.000000     2.000000  1.420000e+09  4.733333e+04
50%      90.000000     2.000000  2.972500e+09  9.908333e+04
75%     122.000000     2.000000  6.181250e+09  2.060417e+05
max    3600.000000     5.000000  9.240000e+10  3.080000e+06
